# 16 · Graph RAG

**Idea:** instead of retrieving by *similarity*, retrieve by *connection*. An LLM extracts `subject | relation | object` triples from each chunk; we build a directed **entity graph** (networkx). At query time we find the entities named in the question, expand to their neighborhood, and return the chunks attached to those nodes — surfacing linked facts a vector search would scatter.

> Extraction runs one LLM call per chunk, so it defaults to Ollama.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

In [ ]:
from rag_lab import load_markdown, GraphRAG

# Use the systems corpus — it is full of named entities and relations.
systems = [c for c in load_markdown() if c.source == 'systems_and_teams.md']
graph = GraphRAG(provider='ollama')
graph.add(systems)
print('nodes:', len(graph), '| edges:', graph.graph.number_of_edges())

Inspect the extracted relationships near an entity — this is the knowledge the LLM pulled out of the prose.

In [ ]:
for s, r, o in graph.edges_near('Atlas', depth=1):
    print(f'{s}  --[{r}]-->  {o}')

Now retrieve by traversal: name an entity in the query and the graph returns the chunks about it **and its neighbors** (`depth=1`).

In [ ]:
for h in graph.search('What is Beacon and who built it?', k=3, depth=1):
    print(f'{h.score:.0f}  {h.chunk.meta.get("heading")}')
    print('   ', h.chunk.text[:120], '...')
    print()

**Takeaway:** graph RAG makes *relationships* first-class. It excels at multi-entity and "how are these connected?" questions where pure similarity search returns disconnected fragments. The cost is an extraction pass over the corpus — done once at index time.

Next → `17_citation_generation.ipynb`